# Intro to Kafka & Spark — the real frameworks

This notebook gives you **awareness of the actual libraries** you'd use in industry:

- **Apache Spark** (via the `pyspark` library) — a distributed *compute* engine. **The Spark cells
  below really run** in "local mode" (a one-machine cluster), so you'll see real Spark output.
- **Apache Kafka** (via the `kafka-python` library) — a distributed *event log* that moves data.
  Kafka needs a running **broker** (a server), so the Kafka cells show the real API but are
  guarded with a flag — flip it on once you have a broker.

> Mental model: **Kafka moves the data, Spark crunches it.** Use them when data outgrows one
> machine; for small data, plain pandas is simpler.

Run the cells top to bottom. The first Spark cell takes ~10–20s to start the engine.

### Setup — install the two libraries

In [1]:
# Install once. pyspark bundles Spark itself (needs Java, already present here).
# kafka-python is a pure-Python client for talking to a Kafka broker.
# (If they're already installed, pip just confirms and moves on.)
import sys, subprocess                                  # to run pip from inside the notebook
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "--break-system-packages", "pyspark", "kafka-python-ng"])  # install both
print("done")                                           # marker so we know the cell finished

done


---
# Part A · Apache Spark (this really runs)

Spark processes large datasets **in parallel** across a cluster. We run a tiny local cluster on
this one machine, which is perfect for learning the API — the *same code* scales to a real cluster.

## A1. Start a SparkSession (the entry point)

`SparkSession` is your handle to Spark. `master="local[2]"` means "run locally using 2 cores"
(a real job would say e.g. `yarn` or a cluster URL instead).

In [2]:
import os                                              # to set an env var for Spark
os.environ["PYSPARK_PYTHON"] = sys.executable           # make Spark use this same Python
from pyspark.sql import SparkSession, functions as F     # SparkSession + the column functions (F)

spark = (SparkSession.builder                            # start building a session
         .appName("intro")                               # a name shown in Spark's UI/logs
         .master("local[2]")                             # run a local 2-core "cluster"
         .config("spark.ui.showConsoleProgress", "false")# quieter output in a notebook
         .config("spark.sql.shuffle.partitions", "4")    # small shuffle width for tiny data
         .getOrCreate())                                  # create it (or reuse if one exists)
spark.sparkContext.setLogLevel("ERROR")                  # hide Spark's chatty INFO/WARN logs
print("Spark version:", spark.version)                   # confirm Spark is up and its version

Spark version: 4.1.2


## A2. Create a DataFrame and look at it

A Spark **DataFrame** is a distributed, table-like structure with named, typed columns — it feels
like pandas or SQL, but the data can be spread across many machines.

In [3]:
# Build a tiny DataFrame from Python rows (normally you'd read files/Kafka/a database)
orders = spark.createDataFrame(
    [("North", "paid",   100),                          # (region, status, amount) rows
     ("South", "paid",   200),
     ("North", "unpaid", 50),
     ("South", "paid",   300),
     ("East",  "paid",   150)],
    ["region", "status", "amount"])                      # the column names

orders.show()                                           # an ACTION: prints the table
orders.printSchema()                                    # show inferred column names + types
print("Row count:", orders.count())                     # count() is also an action

+------+------+------+
|region|status|amount|
+------+------+------+
| North|  paid|   100|
| South|  paid|   200|
| North|unpaid|    50|
| South|  paid|   300|
|  East|  paid|   150|
+------+------+------+

root
 |-- region: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: long (nullable = true)



Row count: 5


## A3. Transformations are *lazy*; actions *trigger*

Transformations (`filter`, `select`, `groupBy`, `withColumn`, ...) just **build a plan** — nothing
runs. An **action** (`show`, `count`, `collect`, `write`) executes the whole optimised plan at once.
`explain()` lets us see the plan before running it.

In [4]:
# These are TRANSFORMATIONS — they return a new DataFrame but compute nothing yet
paid = orders.filter(F.col("status") == "paid")         # keep only paid orders (lazy)
by_region = paid.groupBy("region").agg(                  # group + aggregate (lazy)
    F.sum("amount").alias("revenue"))                    # name the summed column "revenue"

print("Nothing has executed yet. Here is Spark's plan:\n")
by_region.explain()                                     # show the physical plan (the DAG)

print("\nNow we call an ACTION (show) — Spark runs the whole plan:")
by_region.show()                                        # triggers execution

Nothing has executed yet. Here is Spark's plan:



== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[region#0], functions=[sum(amount#2L)])
   +- Exchange hashpartitioning(region#0, 4), ENSURE_REQUIREMENTS, [plan_id=65]
      +- HashAggregate(keys=[region#0], functions=[partial_sum(amount#2L)])
         +- Project [region#0, amount#2L]
            +- Filter (isnotnull(status#1) AND (status#1 = paid))
               +- Scan ExistingRDD[region#0,status#1,amount#2L]



Now we call an ACTION (show) — Spark runs the whole plan:


+------+-------+
|region|revenue|
+------+-------+
| North|    100|
| South|    500|
|  East|    150|
+------+-------+



## A4. Spark SQL — query a DataFrame with SQL

Register a DataFrame as a temporary view and you can query it with plain SQL. Spark optimises SQL
and DataFrame code through the same engine.

In [5]:
orders.createOrReplaceTempView("orders")            # expose the DataFrame as SQL table "orders"

result = spark.sql("""                                  -- ordinary SQL, run by Spark
    SELECT region, SUM(amount) AS revenue                -- total amount per region
    FROM orders
    WHERE status = 'paid'                                -- only paid orders
    GROUP BY region
    ORDER BY revenue DESC                                -- biggest region first
""")
result.show()                                           # action: run the SQL and print

+------+-------+
|region|revenue|
+------+-------+
| South|    500|
|  East|    150|
| North|    100|
+------+-------+



## A5. Read a CSV with Spark

In real life Spark reads from files (CSV, Parquet, JSON), databases, or Kafka. Here we write a
small CSV, then read it back with Spark's `read.csv` (the same call works on a folder of huge files).

In [6]:
# Write a tiny CSV to disk so we have something to read
csv_path = "/tmp/sales.csv"                             # where to put the file
with open(csv_path, "w") as f:                          # open the file for writing
    f.write("region,amount\n")                          # header row
    f.write("North,120\nSouth,80\nNorth,200\nEast,60\n")  # four data rows

df_csv = (spark.read                                    # Spark's reader
          .option("header", True)                       # first line is the header
          .option("inferSchema", True)                  # detect column types automatically
          .csv(csv_path))                                # read the CSV path (could be a folder)

df_csv.show()                                           # show what we loaded
df_csv.groupBy("region").agg(F.avg("amount").alias("avg_amount")).show()  # quick aggregation

+------+------+
|region|amount|
+------+------+
| North|   120|
| South|    80|
| North|   200|
|  East|    60|
+------+------+



+------+----------+
|region|avg_amount|
+------+----------+
| North|     160.0|
|  East|      60.0|
| South|      80.0|
+------+----------+



## A6. Stop the session

Always release the cluster resources when done.

In [7]:
spark.stop()                                         # shut down the Spark session
print("Spark stopped.")                                 # confirmation

Spark stopped.


---
# Part B · Apache Kafka (real API; needs a broker)

Kafka is a **distributed, append-only log** that decouples *producers* (apps that write events)
from *consumers* (apps that read them). Data lives in **topics**, which are split into **partitions**;
each event has an **offset** (its position).

The `kafka-python` client is installed and importable here, but **producing/consuming needs a
running Kafka broker** (a server, usually at `localhost:9092`). So below we import the library to
prove it's available, then show the real producer/consumer code guarded by a flag.

### B1. The library is installed (this runs)

In [8]:
import kafka                                          # the kafka-python client library
from kafka import KafkaProducer, KafkaConsumer            # the two main classes you use
print("kafka-python version:", kafka.__version__)         # confirm it imported
print("Main classes available:", KafkaProducer.__name__, "and", KafkaConsumer.__name__)

kafka-python version: 2.2.3
Main classes available: KafkaProducer and KafkaConsumer


### B2. Producing events (real code, guarded)

This is exactly how you publish events to a topic. It will only run if you have a broker, so we
guard it with `RUN_KAFKA`. Flip it to `True` after starting a broker (see B4).

In [9]:
RUN_KAFKA = False                                    # set True only when a broker is running

if RUN_KAFKA:                                            # guard so this notebook runs without a broker
    import json                                          # to turn dicts into bytes
    producer = KafkaProducer(                            # connect a producer to the broker
        bootstrap_servers="localhost:9092",             # broker address
        value_serializer=lambda v: json.dumps(v).encode())  # serialise each value to JSON bytes
    for i in range(5):                                   # publish 5 example events
        event = {"order_id": i, "amount": 100 + i}       # the event payload
        producer.send("orders", value=event)            # append the event to topic "orders"
    producer.flush()                                     # make sure everything is sent
    producer.close()                                    # close the connection
    print("Sent 5 events to topic 'orders'.")
else:
    print("RUN_KAFKA is False — skipping (no broker). The code above is real; flip the flag to run it.")

RUN_KAFKA is False — skipping (no broker). The code above is real; flip the flag to run it.


### B3. Consuming events (real code, guarded)

A **consumer** reads events from a topic. A `group_id` makes it part of a **consumer group**, which
shares the work and remembers its **offset** so it can resume after a restart.

In [10]:
if RUN_KAFKA:                                         # again, only with a live broker
    import json                                          # to decode JSON bytes back to dicts
    consumer = KafkaConsumer(                            # connect a consumer
        "orders",                                       # the topic to read
        bootstrap_servers="localhost:9092",             # broker address
        group_id="my-group",                            # belong to this consumer group
        auto_offset_reset="earliest",                   # if no saved offset, start from the beginning
        consumer_timeout_ms=3000,                        # stop waiting after 3s of no messages
        value_deserializer=lambda b: json.loads(b.decode()))  # decode JSON bytes to a dict
    for msg in consumer:                                 # iterate over incoming events
        print(f"partition={msg.partition} offset={msg.offset} value={msg.value}")
    consumer.close()                                    # close the connection
else:
    print("RUN_KAFKA is False — skipping (no broker). This is the real consumer API.")

RUN_KAFKA is False — skipping (no broker). This is the real consumer API.


### B4. How to get a broker to actually try this

Kafka is a server, so you run it once, then flip `RUN_KAFKA = True` and re-run B2/B3.

The quickest way locally is **Docker**:

```bash
# starts a single-node Kafka broker on localhost:9092
docker run -d --name kafka -p 9092:9092 apache/kafka:latest
```

Then create the topic (optional — Kafka can auto-create it):

```bash
docker exec kafka /opt/kafka/bin/kafka-topics.sh \
  --create --topic orders --bootstrap-server localhost:9092 \
  --partitions 3 --replication-factor 1
```

Now B2 (produce) and B3 (consume) will work against your local broker.

---
## Wrap-up — what you now know

- **Spark** (`pyspark`): a `SparkSession` is the entry point; **DataFrames** look like SQL/pandas;
  **transformations are lazy** and **actions trigger** the optimised plan; you can use **Spark SQL**
  and read files at scale — and the *same code* runs on a real cluster.
- **Kafka** (`kafka-python`): **producers** publish events to **topics** (split into **partitions**
  with **offsets**); **consumers** in a **group** read and track their position. It's a server, so
  you point the client at a **broker**.
- **Together**: Kafka buffers real-time events; Spark Structured Streaming reads them in
  micro-batches, transforms, and writes results downstream.

**Where to go next**
- Spark docs: the DataFrame API, `groupBy`/`join`/`window`, and Structured Streaming (`readStream`).
- Kafka docs: topics, partitions, consumer groups, and `kafka-python` quickstart.
- Rule of thumb: reach for these only when data truly outgrows one machine.